In [1]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats


# 노트북 안에 그래프 그리기 위해
%matplotlib inline

# 그래프에서 격자로 숫자 범위가 눈에 잘 띄도록 gglot 스타일 사용
plt.style.use('ggplot')

# 그래프에서 마이너스 폰트 깨지는 문제에 대한 대처
mpl.rcParams['axes.unicode_minus'] = False

plt.rcParams['font.family'] = 'NanumGothic'

### 2021-12-17 (막곡 real_weight, analysis_weight Join)

In [6]:
df_Image = pd.read_csv("data/weight_막곡_all.csv")
print(df_Image.shape)

# 실제 무게 데이터 로드
df_real = pd.read_csv("data/chickenweight_막곡_all.csv")
print(df_real.shape)


(1140, 12)
(129382, 11)


In [7]:
weight_df = pd.read_csv('data/chickenweight_막곡_all.csv')
pixel_df = pd.read_csv('data/weight_막곡_all.csv')

weight_df.head(2)
pixel_df.head(2)

,TID,CREATE_TIME,HOUSE_ID,MODULE_ID,DATA_TYPE,ORG_FILE_NAME,WEIGHT_PREDICTION_RESULT_FILE_NAME,WEIGHT_PREDICTION_COUNT,WEIGHT_PREDICTION_PIXEL_MEAN,WEIGHT_PREDICTION_WEIGHT,WEIGHT_PREDICTION_STATUS,SEND_TID
0,24bb28824215,2021-12-15 17:06:18,H02,"CT02,6",real,"H02_CT02,6_20211215170618_farm_image_real_24bb...",NaN,NaN,NaN,NaN,fail,f1c86b8841b0
1,52bdf392452c,2021-12-15 17:06:34,H03,"CT03,6",real,"H03_CT03,6_20211215170634_farm_image_real_52bd...",NaN,NaN,NaN,NaN,fail,24792c474fac


In [8]:
weight_df['CREATE_TIME'] = pd.to_datetime(weight_df.CREATE_TIME, format='%Y-%m-%d %H:%M:%S')
pixel_df['CREATE_TIME'] = pd.to_datetime(pixel_df.CREATE_TIME, format='%Y-%m-%d %H:%M:%S')

In [9]:
weight_df.sort_values('CREATE_TIME', inplace=True)
pixel_df.sort_values('CREATE_TIME', inplace=True)

In [10]:
weight_grp_df = weight_df.groupby(['CREATE_TIME','HOUSE_ID','MODULE_ID'],as_index=False)[['SENSOR_DATA']].mean()
#weight_grp_df[(weight_grp_df.CREATE_TIME >= '2021-12-17 16:37') & (weight_grp_df.HOUSE_ID == 'H01')]
#weight_grp_df[(weight_grp_df.CREATE_TIME >= '2021-12-17 16:47') & (weight_grp_df.HOUSE_ID == 'H01')]
df_01 = pd.merge_asof(pixel_df[pixel_df.HOUSE_ID=='H01'].iloc[:,[1,2,3,4,5,6,7,8,9]], weight_grp_df[weight_grp_df.HOUSE_ID=='H01'], on="CREATE_TIME", direction="nearest")
df_02 = pd.merge_asof(pixel_df[pixel_df.HOUSE_ID=='H02'].iloc[:,[1,2,3,4,5,6,7,8,9]], weight_grp_df[weight_grp_df.HOUSE_ID=='H02'], on="CREATE_TIME", direction="nearest")
df_03 = pd.merge_asof(pixel_df[pixel_df.HOUSE_ID=='H03'].iloc[:,[1,2,3,4,5,6,7,8,9]], weight_grp_df[weight_grp_df.HOUSE_ID=='H03'], on="CREATE_TIME", direction="nearest")
df_04 = pd.merge_asof(pixel_df[pixel_df.HOUSE_ID=='H04'].iloc[:,[1,2,3,4,5,6,7,8,9]], weight_grp_df[weight_grp_df.HOUSE_ID=='H04'], on="CREATE_TIME", direction="nearest")
df_list = [df_01, df_02, df_03, df_04]
df_total = pd.concat(df_list, ignore_index=True)
df_total = df_total[['CREATE_TIME','ORG_FILE_NAME', 'WEIGHT_PREDICTION_PIXEL_MEAN','WEIGHT_PREDICTION_WEIGHT','SENSOR_DATA']]

In [11]:
df_total

,CREATE_TIME,ORG_FILE_NAME,WEIGHT_PREDICTION_PIXEL_MEAN,WEIGHT_PREDICTION_WEIGHT,SENSOR_DATA
0,2021-12-15 17:06:59,"H01_CT01,6_20211215170659_farm_image_real_3d2d...",NaN,NaN,107.799091
1,2021-12-15 17:16:33,"H01_CT01,6_20211215171633_farm_image_real_8ab0...",NaN,NaN,2.677273
2,2021-12-15 17:26:38,"H01_CT01,6_20211215172638_farm_image_real_137e...",NaN,NaN,6.211818
3,2021-12-15 17:36:21,"H01_CT01,6_20211215173621_farm_image_real_6df4...",NaN,NaN,6.921818
4,2021-12-15 17:46:45,"H01_CT01,6_20211215174645_farm_image_real_c382...",NaN,NaN,4.122727
...,...,...,...,...,...
1135,2021-12-17 16:36:43,"H04_CT04,6_20211217163643_farm_image_real_518b...","[2144,2325,2176,2102,2039]",103.8,0.372727
1136,2021-12-17 16:46:46,"H04_CT04,6_20211217164646_farm_image_real_7748...",[2347],113.6,-0.732727
1137,2021-12-17 16:56:48,"H04_CT04,6_20211217165648_farm_image_real_34e2...","[2048,1891]",94.2,1.173636
1138,2021-12-17 17:06:45,"H04_CT04,6_20211217170645_farm_image_real_c102...","[2158,2187,2174]",104.7,-0.400909


In [12]:
df_total.to_csv('real_image_weight_compare.csv', encoding='utf-8')

In [13]:
weight_grp_df = weight_df.groupby(['CREATE_TIME','HOUSE_ID'],as_index=False)[['SENSOR_DATA']].mean()
df_list = []
for i in range(0, len(pixel_df)):
    if i == len(pixel_df) -1 :
        break
    #print(pixel_df['CREATE_TIME'].iloc[i])
    #print(pixel_df['CREATE_TIME'].iloc[i+1])
    #print("---------------------------")
    #weight_grp_df[(weight_grp_df.CREATE_TIME >= pixel_df['CREATE_TIME'][i]) & (weight_grp_df.HOUSE_ID == 'H01')]
    #weight_grp_df[(weight_grp_df.CREATE_TIME >= pixel_df['CREATE_TIME'][i+1]) & (weight_grp_df.HOUSE_ID == 'H01')]
    condition = (weight_grp_df.HOUSE_ID=='H01') & (weight_grp_df.CREATE_TIME >= pixel_df['CREATE_TIME'][i]) & (weight_grp_df.CREATE_TIME >= pixel_df['CREATE_TIME'][i+1])

    new_df = pd.merge_asof(pixel_df[pixel_df.HOUSE_ID=='H01'].iloc[:,[1,2,3,4,5,6,7,8,9]], weight_grp_df[condition], on="CREATE_TIME", direction="nearest")
    new_df = new_df[new_df.SENSOR_DATA.notna()]
    df_list.append(new_df)

df_all = pd.concat(df_list, ignore_index = True)

df_all = df_all[['CREATE_TIME','ORG_FILE_NAME', 'WEIGHT_PREDICTION_PIXEL_MEAN','WEIGHT_PREDICTION_WEIGHT','SENSOR_DATA']]
df_all = df_all[~df_all.WEIGHT_PREDICTION_WEIGHT.isnull()]

#df = df[df['SENSOR_DATA'] >= 0]
#df_H01 = df.reset_index(drop = True)
    

In [14]:
df_all.tail(1000)

,CREATE_TIME,ORG_FILE_NAME,WEIGHT_PREDICTION_PIXEL_MEAN,WEIGHT_PREDICTION_WEIGHT,SENSOR_DATA
319677,2021-12-16 22:46:43,"H01_CT01,6_20211216224643_farm_image_real_9570...","[2272,2101,2590,2035,2005]",106.1,0.731818
319678,2021-12-16 22:56:41,"H01_CT01,6_20211216225641_farm_image_real_6f51...","[2283,2567,2209,2240,2228,2202,2382,2356,2567]",113.1,0.731818
319679,2021-12-16 23:06:34,"H01_CT01,6_20211216230634_farm_image_real_0aa7...","[2493,2330,2256]",114.3,0.731818
319680,2021-12-16 23:17:14,"H01_CT01,6_20211216231714_farm_image_real_66f1...","[2282,2089,2107]",103.9,0.731818
319681,2021-12-16 23:26:46,"H01_CT01,6_20211216232646_farm_image_real_fb16...",[2469],119.9,0.731818
...,...,...,...,...,...
321192,2021-12-17 16:27:18,"H01_CT01,6_20211217162718_farm_image_real_631f...","[2290,2301,2169,1879,2088,1939,2307,2443,2146,...",104.5,223.090000
321193,2021-12-17 16:37:45,"H01_CT01,6_20211217163745_farm_image_real_9a44...","[3154,2457,2837,2483,2682,2694]",132.7,223.090000
321194,2021-12-17 16:47:17,"H01_CT01,6_20211217164717_farm_image_real_cafe...","[2631,2619,2520,2701,2187,2247]",120.6,223.090000
321195,2021-12-17 16:58:02,"H01_CT01,6_20211217165802_farm_image_real_11e7...","[2804,2562,2785]",132.6,223.090000
